In [1]:
import logging

logging.basicConfig(
    filename="cleaning_log.txt",
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    filemode="w",
    encoding="utf-8",
)

logging.info("Task 5 — Financial Hardship & Equity Data Preparation Started")
logging.info(
    "Files used: 07_العبء_المالي_للرعاية_الصحية_حسب_مستوى_الدخل.csv, 08_العبء_المالي_للرعاية_الصحية_ومكوناته.csv, country_dictionary_v1.csv"
)
logging.info(
    "Starting Part 2: Processing 07_العبء_المالي_للرعاية_الصحية_حسب_مستوى_الدخل.csv (Financial Hardship by Wealth Quintile)"
)

In [2]:
import pandas as pd

df_07 = pd.read_csv("07_العبء_المالي_للرعاية_الصحية_حسب_مستوى_الدخل.csv")
print("Columns:", df_07.columns.tolist())
print("Total Raw Rows:", len(df_07))
df_07.head()

Columns: ['IndicatorCode', 'Indicator', 'ValueType', 'ParentLocationCode', 'ParentLocation', 'Location type', 'SpatialDimValueCode', 'Location', 'Period type', 'Period', 'IsLatestYear', 'Dim1 type', 'Dim1', 'Dim1ValueCode', 'Dim2 type', 'Dim2', 'Dim2ValueCode', 'Dim3 type', 'Dim3', 'Dim3ValueCode', 'DataSourceDimValueCode', 'DataSource', 'FactValueNumericPrefix', 'FactValueNumeric', 'FactValueUoM', 'FactValueNumericLowPrefix', 'FactValueNumericLow', 'FactValueNumericHighPrefix', 'FactValueNumericHigh', 'Value', 'FactValueTranslationID', 'FactComments', 'Language', 'DateModified']
Total Raw Rows: 5235


,IndicatorCode,Indicator,ValueType,ParentLocationCode,ParentLocation,Location type,SpatialDimValueCode,Location,Period type,Period,...,FactValueUoM,FactValueNumericLowPrefix,FactValueNumericLow,FactValueNumericHighPrefix,FactValueNumericHigh,Value,FactValueTranslationID,FactComments,Language,DateModified
0,FINANCIALHARDSHIP_PROPORTIONOFPOP,Financial hardship in health and components (S...,numeric,EUR,Europe,Country,MKD,North Macedonia,Year,2024,...,NaN,NaN,NaN,NaN,NaN,12.12,NaN,NaN,EN,2025-11-18T22:00:00.000Z
1,FINANCIALHARDSHIP_PROPORTIONOFPOP,Financial hardship in health and components (S...,numeric,EUR,Europe,Country,GEO,Georgia,Year,2024,...,NaN,NaN,NaN,NaN,NaN,15.28,NaN,NaN,EN,2025-11-18T22:00:00.000Z
2,FINANCIALHARDSHIP_PROPORTIONOFPOP,Financial hardship in health and components (S...,numeric,EUR,Europe,Country,MKD,North Macedonia,Year,2024,...,NaN,NaN,NaN,NaN,NaN,2.30,NaN,NaN,EN,2025-11-18T22:00:00.000Z
3,FINANCIALHARDSHIP_PROPORTIONOFPOP,Financial hardship in health and components (S...,numeric,EUR,Europe,Country,MKD,North Macedonia,Year,2024,...,NaN,NaN,NaN,NaN,NaN,2.30,NaN,NaN,EN,2025-11-18T22:00:00.000Z
4,FINANCIALHARDSHIP_PROPORTIONOFPOP,Financial hardship in health and components (S...,numeric,EUR,Europe,Country,MKD,North Macedonia,Year,2024,...,NaN,NaN,NaN,NaN,NaN,26.54,NaN,NaN,EN,2025-11-18T22:00:00.000Z


In [3]:
print("Indicators:", df_07["Indicator"].unique())
print("Location types:", df_07["Location type"]. value_counts(dropna=False))
print("Number of unique SpatialDimValueCode:", df_07["SpatialDimValueCode"].nunique())
print("Number of Survey Years (Period):", df_07["Period"].nunique())
print("Min Survey Year:", df_07["Period"].min(), "| Max Survey Year:", df_07["Period"].max())
print("\nDim1 type:", df_07["Dim1 type"].unique())
print("Dim1 values (Wealth Quintiles):\n", df_07["Dim1"].value_counts(dropna=False))
print("\nDim2 type:", df_07["Dim2 type"].unique())
print("Dim2 values:\n", df_07["Dim2"].value_counts(dropna=False))
print("\nMissing in FactValueNumeric:", df_07["FactValueNumeric"].isna().sum())

Indicators: ['Financial hardship in health and components (SDG 3.8.2, 2025 definition): population (%)']
Location types: Location type
Country    5235
Name: count, dtype: int64
Number of unique SpatialDimValueCode: 165
Number of Survey Years (Period): 39
Min Survey Year: 1985 | Max Survey Year: 2024

Dim1 type: ['Wealth Quintile']
Dim1 values (Wealth Quintiles):
 Dim1
Q2              1047
Q3              1047
Q4              1047
Q5 (Richest)    1047
Q1 (Poorest)    1047
Name: count, dtype: int64

Dim2 type: [nan]
Dim2 values:
 Dim2
NaN    5235
Name: count, dtype: int64

Missing in FactValueNumeric: 0


In [4]:
logging.info(f"File 07 Raw Rows: {len(df_07)}")
logging.info(f"File 07 Unique Countries (ISO3): {df_07['SpatialDimValueCode'].nunique()}")
logging.info(
    f"File 07 Unique Survey Years: {df_07['Period'].nunique()} (Min: {df_07['Period'].min()}, Max: {df_07['Period'].max()})"
)
logging.info(f"File 07 Wealth Quintiles: {df_07['Dim1'].unique().tolist()}")
logging.info(f"File 07 Raw Missing in FactValueNumeric: {df_07['FactValueNumeric'].isna().sum()}")

dup_07 = df_07.duplicated(subset=["SpatialDimValueCode", "Period", "Dim1"]).sum()
print("Duplicates before Pivot (ISO3 + Survey_Year + Wealth_Quintile):", dup_07)

print("Period dtype:", df_07["Period"].dtype)
print("FactValueNumeric dtype:", df_07["FactValueNumeric"].dtype)
print("Min Value:", df_07["FactValueNumeric"].min(), "| Max Value:", df_07["FactValueNumeric"].max())
print("Values < 0 or > 100:", len(df_07[(df_07["FactValueNumeric"] < 0) | (df_07["FactValueNumeric"] > 100)]))

Duplicates before Pivot (ISO3 + Survey_Year + Wealth_Quintile): 0
Period dtype: int64
FactValueNumeric dtype: float64
Min Value: 0.0 | Max Value: 100.0
Values < 0 or > 100: 0


In [5]:
logging.info(f"File 07 Duplicates before Pivot (ISO3 + Survey_Year + Wealth_Quintile): {dup_07}")
logging.info(
    f"File 07 Value Range: Min={df_07['FactValueNumeric'].min()}, Max={df_07['FactValueNumeric'].max()}"
)

df_07_pivot = df_07.pivot(
    index=["SpatialDimValueCode", "Period"],
    columns="Dim1",
    values="FactValueNumeric",
).reset_index()

df_07_pivot.columns.name = None

df_07_pivot = df_07_pivot.rename(
    columns={
        "SpatialDimValueCode": "ISO3",
        "Period": "Survey_Year",
        "Q1 (Poorest)": "Q1_Poorest_Pct",
        "Q2": "Q2_Pct",
        "Q3": "Q3_Pct",
        "Q4": "Q4_Pct",
        "Q5 (Richest)": "Q5_Richest_Pct",
    }
)

quintile_cols = ["Q1_Poorest_Pct", "Q2_Pct", "Q3_Pct", "Q4_Pct", "Q5_Richest_Pct"]
df_07_pivot = df_07_pivot[["ISO3", "Survey_Year"] + quintile_cols]

df_07_pivot["Equity_Gap_Pct_Points"] = (
    df_07_pivot["Q1_Poorest_Pct"] - df_07_pivot["Q5_Richest_Pct"]
)

equity_calculated = df_07_pivot["Equity_Gap_Pct_Points"].notna().sum()
equity_missing = df_07_pivot["Equity_Gap_Pct_Points"].isna().sum()

print("Rows before Pivot:", len(df_07), "| Rows after Pivot:", len(df_07_pivot))
print("Missing values after Pivot:\n", df_07_pivot[quintile_cols + ["Equity_Gap_Pct_Points"]].isna().sum())
print("Equity_Gap_Pct_Points calculated:", equity_calculated, "| Missing:", equity_missing)

logging.info(f"File 07 Pivot completed: Rows before Pivot = {len(df_07)}, Rows after Pivot = {len(df_07_pivot)}")
logging.info(f"File 07 Missing values after Pivot: {df_07_pivot[quintile_cols].isna().sum().to_dict()}")
logging.info(
    f"Equity_Gap_Pct_Points in File 07: Calculated = {equity_calculated}, Missing (due to missing Q1 or Q5) = {equity_missing}"
)

df_07_pivot.head()

Rows before Pivot: 5235 | Rows after Pivot: 1047
Missing values after Pivot:
 Q1_Poorest_Pct           0
Q2_Pct                   0
Q3_Pct                   0
Q4_Pct                   0
Q5_Richest_Pct           0
Equity_Gap_Pct_Points    0
dtype: int64
Equity_Gap_Pct_Points calculated: 1047 | Missing: 0


,ISO3,Survey_Year,Q1_Poorest_Pct,Q2_Pct,Q3_Pct,Q4_Pct,Q5_Richest_Pct,Equity_Gap_Pct_Points
0,AFG,2007,95.71,23.56,1.90,0.49,0.13,95.58
1,AFG,2013,91.13,92.47,35.73,9.52,5.30,85.83
2,AFG,2016,46.24,57.72,33.44,15.15,8.95,37.29
3,AFG,2020,68.27,58.80,27.98,13.36,4.41,63.86
4,AGO,2008,14.07,23.41,12.01,6.62,7.83,6.24


In [6]:
df_07_pivot["Equity_Gap_Pct_Points"] = df_07_pivot["Equity_Gap_Pct_Points"].round(2)

dup_after_07 = df_07_pivot.duplicated(subset=["ISO3", "Survey_Year"]).sum()
q1_raw_sum = df_07.loc[df_07["Dim1"] == "Q1 (Poorest)", "FactValueNumeric"].sum().round(2)
q5_raw_sum = df_07.loc[df_07["Dim1"] == "Q5 (Richest)", "FactValueNumeric"].sum().round(2)

assert dup_after_07 == 0, "Error: Duplicate ISO3 + Survey_Year found!"
assert df_07_pivot["Q1_Poorest_Pct"].sum().round(2) == q1_raw_sum, "Error: Q1 mismatch!"
assert df_07_pivot["Q5_Richest_Pct"].sum().round(2) == q5_raw_sum, "Error: Q5 mismatch!"

print("File 07 Validation Passed! Duplicates after Pivot:", dup_after_07)
logging.info(
    f"File 07 Validation Passed: Duplicate (ISO3 + Survey_Year) = {dup_after_07}, Q1=Poorest & Q5=Richest verified."
)

File 07 Validation Passed! Duplicates after Pivot: 0


In [7]:
print("=== 1. Columns & Data Types ===")
print(df_07_pivot.dtypes)

print("\n=== 2. Summary Stats (Min / Max / Missing) ===")
print(df_07_pivot.describe().loc[["count", "min", "max"]])

# 3. مقارنة دولة (مثلًا EGY لو موجودة، أو أول دولة) قبل وبعد الـ Pivot
sample_iso = "EGY" if "EGY" in df_07_pivot["ISO3"].values else df_07_pivot["ISO3"].iloc[0]
sample_year = df_07_pivot.loc[df_07_pivot["ISO3"] == sample_iso, "Survey_Year"].iloc[0]

print(f"\n=== 3. Spot Check for {sample_iso} in {sample_year} (BEFORE Pivot) ===")
print(
    df_07[(df_07["SpatialDimValueCode"] == sample_iso) & (df_07["Period"] == sample_year)][
        ["SpatialDimValueCode", "Period", "Dim1", "FactValueNumeric"]
    ]
)

print(f"\n=== 3. Spot Check for {sample_iso} in {sample_year} (AFTER Pivot) ===")
print(df_07_pivot[(df_07_pivot["ISO3"] == sample_iso) & (df_07_pivot["Survey_Year"] == sample_year)])

print("\n=== 4. Current Contents of cleaning_log.txt ===")
with open("cleaning_log.txt", "r", encoding="utf-8") as f:
    print(f.read())

=== 1. Columns & Data Types ===
ISO3                      object
Survey_Year                int64
Q1_Poorest_Pct           float64
Q2_Pct                   float64
Q3_Pct                   float64
Q4_Pct                   float64
Q5_Richest_Pct           float64
Equity_Gap_Pct_Points    float64
dtype: object

=== 2. Summary Stats (Min / Max / Missing) ===
       Survey_Year  Q1_Poorest_Pct  Q2_Pct  Q3_Pct   Q4_Pct  Q5_Richest_Pct  \
count       1047.0          1047.0  1047.0  1047.0  1047.00         1047.00   
min         1985.0             0.0     0.0     0.0     0.00            0.00   
max         2024.0           100.0   100.0   100.0    90.93           45.76   

       Equity_Gap_Pct_Points  
count                1047.00  
min                   -28.91  
max                    99.75  

=== 3. Spot Check for EGY in 1997 (BEFORE Pivot) ===
     SpatialDimValueCode  Period          Dim1  FactValueNumeric
4994                 EGY    1997            Q4              2.14
5002             

In [8]:
logging.info(
    "Starting Part 1: Processing 08_العبء_المالي_للرعاية_الصحية_ومكوناته.csv (Financial Hardship Components)"
)

df_08 = pd.read_csv("08_العبء_المالي_للرعاية_الصحية_ومكوناته.csv")

print("Total Raw Rows:", len(df_08))
print("Indicators:", df_08["Indicator"].unique())
print("Location types:\n", df_08["Location type"].value_counts(dropna=False))
print("Number of unique SpatialDimValueCode:", df_08["SpatialDimValueCode"].nunique())
print("Number of Survey Years (Period):", df_08["Period"].nunique())
print("Min Survey Year:", df_08["Period"].min(), "| Max Survey Year:", df_08["Period"].max())
print("\nDim1 type:", df_08["Dim1 type"].unique())
print("Dim1 values (Components):\n", df_08["Dim1"].value_counts(dropna=False))
print("\nDim2 type:", df_08["Dim2 type"].unique())
print("Dim2 values:\n", df_08["Dim2"].value_counts(dropna=False))
print("\nMissing in FactValueNumeric:", df_08["FactValueNumeric"].isna().sum())

Total Raw Rows: 3218
Indicators: ['Financial hardship in health and components (SDG 3.8.2, 2025 definition): population (%)']
Location types:
 Location type
Country    3218
Name: count, dtype: int64
Number of unique SpatialDimValueCode: 168
Number of Survey Years (Period): 39
Min Survey Year: 1985 | Max Survey Year: 2024

Dim1 type: ['Financial hardship component']
Dim1 values (Components):
 Dim1
Total                                   1088
Impoverishing spending (Component 2)    1065
Large spending (Component 1)            1065
Name: count, dtype: int64

Dim2 type: [nan]
Dim2 values:
 Dim2
NaN    3218
Name: count, dtype: int64

Missing in FactValueNumeric: 0


In [9]:
logging.info(f"File 08 Raw Rows: {len(df_08)}")
logging.info(f"File 08 Unique Countries (ISO3): {df_08['SpatialDimValueCode'].nunique()}")
logging.info(
    f"File 08 Unique Survey Years: {df_08['Period'].nunique()} (Min: {df_08['Period'].min()}, Max: {df_08['Period'].max()})"
)
logging.info(f"File 08 Financial Hardship Components: {df_08['Dim1'].unique().tolist()}")
logging.info(f"File 08 Raw Missing in FactValueNumeric: {df_08['FactValueNumeric'].isna().sum()}")

dup_08 = df_08.duplicated(subset=["SpatialDimValueCode", "Period", "Dim1"]).sum()
print("Duplicates before Pivot (ISO3 + Survey_Year + Component):", dup_08)

print("Period dtype:", df_08["Period"].dtype)
print("FactValueNumeric dtype:", df_08["FactValueNumeric"].dtype)
print("Min Value:", df_08["FactValueNumeric"].min(), "| Max Value:", df_08["FactValueNumeric"].max())
print("Values < 0 or > 100:", len(df_08[(df_08["FactValueNumeric"] < 0) | (df_08["FactValueNumeric"] > 100)]))

Duplicates before Pivot (ISO3 + Survey_Year + Component): 0
Period dtype: int64
FactValueNumeric dtype: float64
Min Value: 0.0 | Max Value: 77.45
Values < 0 or > 100: 0


In [10]:
logging.info(f"File 08 Duplicates before Pivot (ISO3 + Survey_Year + Component): {dup_08}")
logging.info(
    f"File 08 Value Range: Min={df_08['FactValueNumeric'].min()}, Max={df_08['FactValueNumeric'].max()}"
)

# 2. تنفيذ الـ Pivot بدون أي Aggregation
df_08_pivot = df_08.pivot(
    index=["SpatialDimValueCode", "Period"],
    columns="Dim1",
    values="FactValueNumeric",
).reset_index()

df_08_pivot.columns.name = None

df_08_pivot = df_08_pivot.rename(
    columns={
        "SpatialDimValueCode": "ISO3",
        "Period": "Survey_Year",
        "Total": "Financial_Hardship_Total_Pct",
        "Large spending (Component 1)": "Large_Spending_Pct",
        "Impoverishing spending (Component 2)": "Impoverishing_Spending_Pct",
    }
)

comp_cols = [
    "Financial_Hardship_Total_Pct",
    "Large_Spending_Pct",
    "Impoverishing_Spending_Pct",
]
df_08_pivot = df_08_pivot[["ISO3", "Survey_Year"] + comp_cols]

print("Rows before Pivot:", len(df_08), "| Rows after Pivot:", len(df_08_pivot))
print("Missing values after Pivot:\n", df_08_pivot[comp_cols].isna().sum())

logging.info(f"File 08 Pivot completed: Rows before Pivot = {len(df_08)}, Rows after Pivot = {len(df_08_pivot)}")
logging.info(f"File 08 Missing values after Pivot: {df_08_pivot[comp_cols].isna().sum().to_dict()}")

dup_after_08 = df_08_pivot.duplicated(subset=["ISO3", "Survey_Year"]).sum()
assert dup_after_08 == 0, "Error: Duplicate ISO3 + Survey_Year found in File 08!"
assert (
    df_08_pivot["Financial_Hardship_Total_Pct"].sum().round(2)
    == df_08.loc[df_08["Dim1"] == "Total", "FactValueNumeric"].sum().round(2)
)
assert (
    df_08_pivot["Large_Spending_Pct"].sum().round(2)
    == df_08.loc[df_08["Dim1"] == "Large spending (Component 1)", "FactValueNumeric"].sum().round(2)
)
assert (
    df_08_pivot["Impoverishing_Spending_Pct"].sum().round(2)
    == df_08.loc[df_08["Dim1"] == "Impoverishing spending (Component 2)", "FactValueNumeric"].sum().round(2)
)

print("File 08 Validation Passed! Duplicates after Pivot:", dup_after_08)
logging.info(
    f"File 08 Validation Passed: Duplicate (ISO3 + Survey_Year) = {dup_after_08}, all 3 components verified."
)

df_08_pivot.head()

Rows before Pivot: 3218 | Rows after Pivot: 1088
Missing values after Pivot:
 Financial_Hardship_Total_Pct     0
Large_Spending_Pct              23
Impoverishing_Spending_Pct      23
dtype: int64
File 08 Validation Passed! Duplicates after Pivot: 0


,ISO3,Survey_Year,Financial_Hardship_Total_Pct,Large_Spending_Pct,Impoverishing_Spending_Pct
0,AFG,2007,24.36,NaN,NaN
1,AFG,2013,46.84,NaN,NaN
2,AFG,2016,32.30,NaN,NaN
3,AFG,2020,34.56,NaN,NaN
4,AGO,2008,12.79,5.15,7.63


In [11]:
# 1. دمج الملفين (Outer Merge للحفاظ على جميع المسوح في الملفين)
df_merged = pd.merge(
    df_08_pivot,
    df_07_pivot,
    on=["ISO3", "Survey_Year"],
    how="outer",
).sort_values(["ISO3", "Survey_Year"]).reset_index(drop=True)

print("Rows in File 08 (after Pivot):", len(df_08_pivot))
print("Rows in File 07 (after Pivot):", len(df_07_pivot))
print("Rows after Outer Merge:", len(df_merged))
print("Duplicate (ISO3 + Survey_Year) after Merge:", df_merged.duplicated(subset=["ISO3", "Survey_Year"]).sum())

logging.info(
    f"Outer Merge completed on ['ISO3', 'Survey_Year']: File 08 Rows = {len(df_08_pivot)}, File 07 Rows = {len(df_07_pivot)}, Merged Rows = {len(df_merged)}"
)

# 2. قراءة ملف قاموس الدول لفحص أعمدته
country_dict = pd.read_csv("country_dictionary_v1.csv")
print("\ncountry_dictionary_v1.csv Columns:", country_dict.columns.tolist())
print("Total Countries in Dictionary:", len(country_dict))
country_dict.head()

Rows in File 08 (after Pivot): 1088
Rows in File 07 (after Pivot): 1047
Rows after Outer Merge: 1088
Duplicate (ISO3 + Survey_Year) after Merge: 0

country_dictionary_v1.csv Columns: ['ISO3', 'M49', 'Country', 'WB_Name', 'WHO_Name', 'GHED_Name', 'Nurses_Source_Name', 'Region', 'Income_Group', 'In_GHED', 'In_WHO_Doctors', 'In_WorldBank', 'In_Nurses_Source']
Total Countries in Dictionary: 196


,ISO3,M49,Country,WB_Name,WHO_Name,GHED_Name,Nurses_Source_Name,Region,Income_Group,In_GHED,In_WHO_Doctors,In_WorldBank,In_Nurses_Source
0,AFG,4,Afghanistan,Afghanistan,Afghanistan,Afghanistan,Afghanistan,EMR,Low,True,True,True,True
1,AGO,24,Angola,Angola,Angola,Angola,Angola,AFR,Lower-middle,True,True,True,True
2,ALB,8,Albania,Albania,Albania,Albania,Albania,EUR,Upper-middle,True,True,True,True
3,AND,20,Andorra,Andorra,Andorra,Andorra,Andorra,EUR,High,True,True,True,True
4,ARE,784,United Arab Emirates,United Arab Emirates,United Arab Emirates,United Arab Emirates,United Arab Emirates,EMR,High,True,True,True,True


In [12]:
# 1. تنظيف عمود ISO3 في قاموس الدول والتأكد من عدم وجود تكرار فيه
country_dict["ISO3"] = country_dict["ISO3"].astype(str).str.strip()
assert country_dict["ISO3"].duplicated().sum() == 0, "Duplicate ISO3 in country_dictionary_v1.csv!"

# 2. ربط جدول البيانات المدمج بقاموس الدول باستخدام ISO3 فقط
df_final = pd.merge(
    df_merged,
    country_dict[["ISO3", "Country"]],
    on="ISO3",
    how="left",
)

# 3. فحص الأكواد التي نجح وفشل الـ Mapping الخاص بها
mapped_iso = sorted(df_final.loc[df_final["Country"].notna(), "ISO3"].unique().tolist())
unmapped_iso = sorted(df_final.loc[df_final["Country"].isna(), "ISO3"].unique().tolist())

print(f"Successfully mapped Country Codes ({len(mapped_iso)}):", mapped_iso)
print(f"\nFailed / Unmapped Country Codes ({len(unmapped_iso)}):", unmapped_iso)

# لو ظهر أي كود غير موجود في القاموس، نعرض اسمه الأصلي من الملف الخام عشان تراجعه
if len(unmapped_iso) > 0:
    raw_names = (
        pd.concat([
            df_08[["SpatialDimValueCode", "Location"]],
            df_07[["SpatialDimValueCode", "Location"]],
        ])
        .drop_duplicates()
        .query("SpatialDimValueCode in @unmapped_iso")
    )
    print("\nUnmapped ISO3 details in Raw Data:\n", raw_names)

# 4. ترتيب الأعمدة الـ 12 النهائية بالضبط كما هو مطلوب في التاسك
final_cols = [
    "ISO3",
    "Country",
    "Survey_Year",
    "Financial_Hardship_Total_Pct",
    "Large_Spending_Pct",
    "Impoverishing_Spending_Pct",
    "Q1_Poorest_Pct",
    "Q2_Pct",
    "Q3_Pct",
    "Q4_Pct",
    "Q5_Richest_Pct",
    "Equity_Gap_Pct_Points",
]
df_final = df_final[final_cols].sort_values(["ISO3", "Survey_Year"]).reset_index(drop=True)

# 5. تسجيل نتائج الـ Mapping والـ Missing النهائي في اللوج
logging.info(f"Country Mapping Successful Codes Count: {len(mapped_iso)} | Codes: {mapped_iso}")
logging.info(f"Country Mapping Failed Codes Count: {len(unmapped_iso)} | Codes: {unmapped_iso}")

df_final.head()

Successfully mapped Country Codes (165): ['AFG', 'AGO', 'ALB', 'AND', 'ARG', 'ARM', 'AUT', 'AZE', 'BDI', 'BEL', 'BEN', 'BFA', 'BGD', 'BGR', 'BIH', 'BLR', 'BLZ', 'BOL', 'BRA', 'BRB', 'BTN', 'BWA', 'CAF', 'CHE', 'CHL', 'CHN', 'CIV', 'CMR', 'COD', 'COG', 'COK', 'COL', 'COM', 'CPV', 'CRI', 'CYP', 'CZE', 'DEU', 'DJI', 'DNK', 'DOM', 'ECU', 'EGY', 'ESP', 'EST', 'ETH', 'FIN', 'FSM', 'GAB', 'GBR', 'GEO', 'GHA', 'GIN', 'GMB', 'GNB', 'GRC', 'GRD', 'GTM', 'HND', 'HRV', 'HTI', 'HUN', 'IDN', 'IND', 'IRL', 'IRN', 'IRQ', 'ISL', 'ISR', 'ITA', 'JAM', 'JOR', 'JPN', 'KAZ', 'KEN', 'KGZ', 'KHM', 'KIR', 'KNA', 'KOR', 'LAO', 'LBN', 'LBR', 'LCA', 'LKA', 'LSO', 'LTU', 'LUX', 'LVA', 'MAR', 'MDA', 'MDG', 'MDV', 'MEX', 'MHL', 'MKD', 'MLI', 'MLT', 'MMR', 'MNE', 'MNG', 'MOZ', 'MUS', 'MWI', 'MYS', 'NAM', 'NER', 'NGA', 'NIC', 'NIU', 'NLD', 'NOR', 'NPL', 'NRU', 'OMN', 'PAK', 'PAN', 'PER', 'PHL', 'POL', 'PRT', 'PRY', 'PSE', 'ROU', 'RUS', 'RWA', 'SDN', 'SEN', 'SLB', 'SLE', 'SLV', 'SOM', 'SRB', 'SSD', 'STP', 'SUR', 'SVK',

,ISO3,Country,Survey_Year,Financial_Hardship_Total_Pct,Large_Spending_Pct,Impoverishing_Spending_Pct,Q1_Poorest_Pct,Q2_Pct,Q3_Pct,Q4_Pct,Q5_Richest_Pct,Equity_Gap_Pct_Points
0,AFG,Afghanistan,2007,24.36,NaN,NaN,95.71,23.56,1.90,0.49,0.13,95.58
1,AFG,Afghanistan,2013,46.84,NaN,NaN,91.13,92.47,35.73,9.52,5.30,85.83
2,AFG,Afghanistan,2016,32.30,NaN,NaN,46.24,57.72,33.44,15.15,8.95,37.29
3,AFG,Afghanistan,2020,34.56,NaN,NaN,68.27,58.80,27.98,13.36,4.41,63.86
4,AGO,Angola,2008,12.79,5.15,7.63,14.07,23.41,12.01,6.62,7.83,6.24


In [13]:
unmapped_rows = df_final[df_final["ISO3"].isin(["TKL", "WLF", "XKX"])]
print(f"Total rows for unmapped codes: {len(unmapped_rows)}")
unmapped_rows

Total rows for unmapped codes: 9


,ISO3,Country,Survey_Year,Financial_Hardship_Total_Pct,Large_Spending_Pct,Impoverishing_Spending_Pct,Q1_Poorest_Pct,Q2_Pct,Q3_Pct,Q4_Pct,Q5_Richest_Pct,Equity_Gap_Pct_Points
959,TKL,NaN,2015,1.28,0.00,1.28,6.08,0.00,0.00,0.00,0.00,6.08
1065,WLF,NaN,2005,0.12,0.00,0.12,0.58,0.00,0.00,0.00,0.00,0.58
1067,XKX,NaN,2003,14.05,2.78,11.27,38.81,24.85,4.22,0.80,1.44,37.37
1068,XKX,NaN,2005,17.13,3.79,13.34,45.41,30.68,4.15,4.32,1.03,44.38
1069,XKX,NaN,2006,15.55,3.17,12.38,45.11,25.67,4.89,1.70,0.33,44.78
1070,XKX,NaN,2009,16.17,3.95,12.21,45.67,26.60,4.79,2.89,0.78,44.89
1071,XKX,NaN,2010,17.36,3.42,13.94,51.19,27.51,4.82,1.36,1.90,49.29
1072,XKX,NaN,2011,13.63,2.87,10.76,47.00,14.99,3.02,1.53,1.50,45.50
1073,XKX,NaN,2016,13.92,3.99,9.93,52.41,10.57,1.96,2.32,2.31,50.10


In [14]:
# 1. التحقق من شروط الـ Validation النهائية
final_dup = df_final.duplicated(subset=["ISO3", "Survey_Year"]).sum()
assert final_dup == 0, "Error: Duplicate ISO3 + Survey_Year in final output!"
assert pd.api.types.is_integer_dtype(df_final["Survey_Year"]), "Survey_Year must be integer!"

indicator_cols = [
    "Financial_Hardship_Total_Pct",
    "Large_Spending_Pct",
    "Impoverishing_Spending_Pct",
    "Q1_Poorest_Pct",
    "Q2_Pct",
    "Q3_Pct",
    "Q4_Pct",
    "Q5_Richest_Pct",
    "Equity_Gap_Pct_Points",
]
for col in indicator_cols:
    assert pd.api.types.is_numeric_dtype(df_final[col]), f"{col} must be numeric!"

# 2. حساب إحصائيات الـ Missing والـ Equity Gap في الجدول النهائي المدمج
final_missing = df_final.isna().sum().to_dict()
eq_calc_final = int(df_final["Equity_Gap_Pct_Points"].notna().sum())
eq_miss_final = int(df_final["Equity_Gap_Pct_Points"].isna().sum())

print("=== Final Dataset Validation Summary ===")
print("Final Rows:", len(df_final), "| Final Columns:", len(df_final.columns))
print("Duplicate (ISO3 + Survey_Year):", final_dup)
print("Total Unique ISO3:", df_final["ISO3"].nunique(), "| Mapped Countries:", df_final["Country"].nunique())
print("Survey Years Count:", df_final["Survey_Year"].nunique(), f"(Min: {df_final['Survey_Year'].min()}, Max: {df_final['Survey_Year'].max()})")
print("Equity_Gap_Pct_Points Calculated:", eq_calc_final, "| Not Calculated (Missing Q1/Q5):", eq_miss_final)
print("\nMissing Values per Column in Final Dataset:")
print(df_final.isna().sum())

# 3. توثيق كل النتائج النهائية في cleaning_log.txt
logging.info(
    "Country Mapping Problems Details: 3 ISO3 codes not found in country_dictionary_v1.csv -> "
    "XKX (Kosovo, 7 rows), TKL (Tokelau, 1 row), WLF (Wallis and Futuna Islands, 1 row). "
    "No manual mapping performed per instructions; Country left as NaN."
)
logging.info(f"Final Dataset Shape: {df_final.shape[0]} rows, {df_final.shape[1]} columns")
logging.info(f"Final Validation - Duplicate (ISO3 + Survey_Year): {final_dup}")
logging.info(f"Final Missing Values per Column: {final_missing}")
logging.info(
    f"Final Equity_Gap_Pct_Points Summary: Calculated = {eq_calc_final}, "
    f"Not calculated due to missing Q1/Q5 (surveys present in File 08 only) = {eq_miss_final}"
)

# 4. حفظ الملف النهائي المطلوب للتسليم
df_final.to_csv("financial_access_equity_clean.csv", index=False, encoding="utf-8")
logging.info("Saved final clean dataset to financial_access_equity_clean.csv successfully.")
logging.info("Task 5 — Financial Hardship & Equity Data Preparation Completed Successfully.")
print("\nSaved 'financial_access_equity_clean.csv' successfully!")

=== Final Dataset Validation Summary ===
Final Rows: 1088 | Final Columns: 12
Duplicate (ISO3 + Survey_Year): 0
Total Unique ISO3: 168 | Mapped Countries: 165
Survey Years Count: 39 (Min: 1985, Max: 2024)
Equity_Gap_Pct_Points Calculated: 1047 | Not Calculated (Missing Q1/Q5): 41

Missing Values per Column in Final Dataset:
ISO3                             0
Country                          9
Survey_Year                      0
Financial_Hardship_Total_Pct     0
Large_Spending_Pct              23
Impoverishing_Spending_Pct      23
Q1_Poorest_Pct                  41
Q2_Pct                          41
Q3_Pct                          41
Q4_Pct                          41
Q5_Richest_Pct                  41
Equity_Gap_Pct_Points           41
dtype: int64

Saved 'financial_access_equity_clean.csv' successfully!
